In [ ]:
import multiprocessing
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)

import jax
import jax.numpy as jnp
import jax.random as jr
from blackjax.adaptation.laps import laps
from jaxtyping import PRNGKeyArray
from src.fdm import EMechanismFDMSolver
from src.params import EMechanismFDMParams
from src.utils import generate_noisy_samples
from src.voltammetry import LinearSweepDC

key = jr.key(0)
generate_key, laps_key, key = jr.split(key, 3)

voltammetry = LinearSweepDC()
fdm_solver = EMechanismFDMSolver(voltammetry)

true_params = EMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    E0=jnp.array(2.0),
    dB=jnp.array(1.2),
)

base_current = fdm_solver.solve(true_params)

samples = generate_noisy_samples(
    10,
    base_current,
    0.1,
    key=generate_key,
)


def logdensity_fn(params: EMechanismFDMParams, samples=samples):
    current = fdm_solver.solve(params)
    return -jnp.sum((samples - current) ** 2)


def sample_init(key: PRNGKeyArray) -> EMechanismFDMParams:
    key_alpha, key_K0, key_E0, key_dB = jr.split(key, 4)
    alpha = jr.uniform(key_alpha, minval=0.3, maxval=0.7)
    K0 = jr.uniform(key_alpha, minval=5.0, maxval=50.0)
    E0 = jr.uniform(key_alpha, minval=0.0, maxval=3.0)
    dB = jr.uniform(key_alpha, minval=0.5, maxval=1.5)
    return EMechanismFDMParams(alpha=alpha, K0=K0, E0=E0, dB=dB)


num_chains = 100
num_steps1, num_steps2 = 100, 100

mesh = jax.sharding.Mesh(jax.devices()[:1], "chains")

print("Number of devices: ", len(jax.devices()))

info, grads_per_step, _acc_prob, samples = laps(
    logdensity_fn=logdensity_fn,
    sample_init=sample_init,
    ndims=4,
    num_steps1=num_steps1,
    num_steps2=num_steps2,
    num_chains=num_chains,
    mesh=mesh,
    rng_key=laps_key,
    early_stop=False,
    diagonal_preconditioning=True,
    steps_per_sample=15,
    r_end=0.01,
    diagnostics=False,
    superchain_size=1,
)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(samples.position.alpha)
plt.show()

In [ ]:
import multiprocessing
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)

import blackjax
import jax
import jax.numpy as jnp
import jax.random as jr
import optax
from blackjax.adaptation.laps import laps
from jaxtyping import PRNGKeyArray
from src.fdm import EMechanismFDMSolver
from src.params import EMechanismFDMParams
from src.utils import generate_noisy_samples
from src.voltammetry import LinearSweepDC

NUM_CPUS = multiprocessing.cpu_count()

key = jr.key(0)
generate_key, laps_key, key = jr.split(key, 3)

voltammetry = LinearSweepDC()
fdm_solver = EMechanismFDMSolver(voltammetry)

true_params = EMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    E0=jnp.array(2.0),
    dB=jnp.array(1.2),
)

base_current = fdm_solver.solve(true_params)

samples = generate_noisy_samples(
    10,
    base_current,
    0.1,
    key=generate_key,
)


def logdensity_fn(params: EMechanismFDMParams, samples=samples):
    current = fdm_solver.solve(params)
    return -jnp.sum((samples - current) ** 2)


learning_rate = 1e-3
init_params = EMechanismFDMParams(
    alpha=jnp.linspace(0.5, 0.7, NUM_CPUS),
    K0=jnp.linspace(5.0, 15.0, NUM_CPUS),
    E0=jnp.linspace(1.5, 2.5, NUM_CPUS),
    dB=jnp.linspace(0.8, 1.4, NUM_CPUS),
)

warmup = blackjax.chees_adaptation(logdensity_fn, NUM_CPUS, max_leapfrog_steps=200)
key_warmup, key_sample = jr.split(laps_key)
optim = optax.adam(learning_rate)
(last_states, parameters), _ = warmup.run(
    key_warmup,
    init_params,  # PyTree where each leaf has shape (num_chains, ...)
    1e-2,
    optim,
    100,
)
kernel = blackjax.dynamic_hmc(logdensity_fn, **parameters).step

key_sample = jr.split(key_sample, NUM_CPUS)
new_states, info = jax.vmap(kernel)(key_sample, last_states)


In [ ]:
def inference_loop(
    key,
    kernel,
    initial_state,
    num_samples,
):
    @jax.jit
    def scan_step(state, step_key):
        state, info = kernel(step_key, state)
        return state, (state, info)

    keys = jr.split(key, num_samples)
    _, (states, infos) = jax.lax.scan(scan_step, initial_state, keys)

    return states, infos


inference_loop_multiple_chains = jax.pmap(
    inference_loop, in_axes=(0, None, 0, None), static_broadcasted_argnums=(1, 3)
)

In [ ]:
states, info = inference_loop_multiple_chains(key_sample, kernel, new_states, 4000)
states.position.alpha.block_until_ready()

In [ ]:
import matplotlib.pyplot as plt

options = {"density": True, "bins": 100}

plt.hist(states.position.alpha.flatten(), **options)
plt.show()
print(blackjax.diagnostics.potential_scale_reduction(states.position.alpha))

plt.hist(states.position.K0.flatten(), **options)
plt.show()
print(blackjax.diagnostics.potential_scale_reduction(states.position.K0))

plt.hist(states.position.E0.flatten(), **options)
plt.show()
print(blackjax.diagnostics.potential_scale_reduction(states.position.E0))

plt.hist(states.position.dB.flatten(), **options)
plt.show()
print(blackjax.diagnostics.potential_scale_reduction(states.position.dB))